# Chapter 5: Model Evaluation and Improvement
## *Introduction to Machine Learning with Python*  Müller & Guido

---
> **Ringkasan Bab:** Bagaimana kita tahu model kita benar-benar baik? Bab ini membahas teknik evaluasi yang robust (cross-validation), pemilihan hyperparameter (GridSearchCV), dan berbagai metrik evaluasi yang tepat untuk berbagai jenis masalah.

## 5.1 Cross-Validation

**Masalah dengan train-test split tunggal:**
- Hasil sangat bergantung pada pembagian data yang kebetulan
- Jika data sedikit, test set terlalu kecil → estimasi tidak stabil

**Cross-Validation (CV)** membagi data menjadi k bagian (*folds*):
- Setiap fold digunakan sebagai test set **satu kali**
- Model dilatih pada k-1 fold sisanya
- Akurasi akhir = rata-rata dari k akurasi

**Keuntungan:**
- Estimasi performa yang lebih reliable dan stabil
- Semua data digunakan untuk training DAN testing

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.model_selection import (
    cross_val_score, KFold, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV, train_test_split
)
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

cancer = load_breast_cancer()

# Cross-validation 5-fold
svm = SVC(kernel='rbf', C=1.0)
cv_scores = cross_val_score(svm, cancer.data, cancer.target, cv=5)

print("Skor tiap fold:", cv_scores.round(3))
print(f"Mean akurasi  : {cv_scores.mean():.3f}")
print(f"Std akurasi   : {cv_scores.std():.3f}")
print(f"Range         : [{cv_scores.min():.3f}, {cv_scores.max():.3f}]")

In [ ]:
# Bandingkan: 5-fold vs 10-fold vs Leave-One-Out
from sklearn.model_selection import LeaveOneOut

svm = SVC(kernel='rbf', C=1.0)

for cv in [3, 5, 10]:
    scores = cross_val_score(svm, cancer.data, cancer.target, cv=cv)
    print(f"CV={cv:2d} fold | Mean: {scores.mean():.4f} | Std: {scores.std():.4f}")

### 5.1.1 Stratified K-Fold

Untuk dataset **tidak seimbang** (imbalanced), gunakan **StratifiedKFold** memastikan proporsi kelas di setiap fold sama dengan proporsi aslinya.

In [ ]:
from sklearn.model_selection import StratifiedKFold

# cross_val_score pada klasifikasi sudah otomatis menggunakan StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Distribusi kelas di setiap fold (StratifiedKFold):")
print(f"{'Fold':>4} | {'Malignant':>10} | {'Benign':>8}")
print("-" * 32)
for fold, (train_idx, test_idx) in enumerate(skf.split(cancer.data, cancer.target), 1):
    y_fold = cancer.target[test_idx]
    print(f"{fold:4d} | {(y_fold==0).sum():10d} | {(y_fold==1).sum():8d}")

## 5.2 Hyperparameter Tuning

**Hyperparameter** = parameter yang ditentukan sebelum training (bukan dipelajari dari data).

Contoh:
- SVM: `C`, `gamma`, `kernel`
- Random Forest: `n_estimators`, `max_depth`, `max_features`
- k-NN: `n_neighbors`

**Strategi tuning:**
1. **Grid Search:** coba semua kombinasi dalam grid yang ditentukan
2. **Random Search:** sampling kombinasi secara acak
3. **Bayesian Optimization:** gunakan hasil sebelumnya untuk menentukan titik berikutnya

### 5.2.1 GridSearchCV

Mencoba **semua kombinasi** parameter yang ditentukan, dievaluasi dengan cross-validation.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Grid parameter yang akan dicoba
param_grid = {
    'C'    : [0.001, 0.01, 0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1, 10, 100],
}

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=0
)

# Normalisasi
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

grid_search = GridSearchCV(
    SVC(),
    param_grid,
    cv=5,
    return_train_score=True
)
grid_search.fit(X_train_s, y_train)

print("Parameter terbaik :", grid_search.best_params_)
print(f"Akurasi CV terbaik: {grid_search.best_score_:.3f}")
print(f"Akurasi test      : {grid_search.score(X_test_s, y_test):.3f}")

In [ ]:
# Visualisasi heatmap hasil grid search
import pandas as pd

results = pd.DataFrame(grid_search.cv_results_)
scores = results.pivot_table(
    values='mean_test_score',
    index='param_C',
    columns='param_gamma'
)

plt.figure(figsize=(8, 6))
plt.imshow(scores, cmap='viridis', aspect='auto')
plt.colorbar(label='CV Accuracy')
plt.xticks(range(len(scores.columns)), scores.columns, rotation=45)
plt.yticks(range(len(scores.index)), scores.index)
plt.xlabel('gamma')
plt.ylabel('C')
plt.title('GridSearchCV Heatmap — SVM (C vs gamma)')
plt.tight_layout()
plt.show()

### 5.2.2 RandomizedSearchCV

Lebih efisien dari Grid Search untuk ruang parameter yang besar  **menyample kombinasi secara acak**.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Distribusi parameter untuk random sampling
param_dist = {
    'n_estimators': randint(10, 200),
    'max_depth'   : randint(2, 20),
    'max_features': uniform(0.1, 0.9),
}

rand_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=30,         # coba 30 kombinasi acak
    cv=5,
    random_state=42
)
rand_search.fit(X_train, y_train)

print("Parameter terbaik :", rand_search.best_params_)
print(f"Akurasi CV terbaik: {rand_search.best_score_:.3f}")
print(f"Akurasi test      : {rand_search.score(X_test, y_test):.3f}")

## 5.3 Metrik Evaluasi

**Akurasi saja tidak cukup**, terutama untuk dataset tidak seimbang.

Contoh: Dataset dengan 95% kelas negatif → model yang selalu prediksi negatif = akurasi 95%! Tapi tidak berguna.

### Confusion Matrix

|  | Prediksi Positif | Prediksi Negatif |
|---|---|---|
| **Aktual Positif** | TP (True Positive) | FN (False Negative) |
| **Aktual Negatif** | FP (False Positive) | TN (True Negative) |

In [ ]:
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve
)
from sklearn.svm import SVC

# Train model
svm_best = SVC(C=grid_search.best_params_['C'],
               gamma=grid_search.best_params_['gamma'])
svm_best.fit(X_train_s, y_train)
y_pred = svm_best.predict(X_test_s)

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap='Blues')
plt.colorbar()
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i,j], ha='center', va='center',
                 fontsize=20, color='white' if cm[i,j] > cm.max()/2 else 'black')
plt.xticks([0,1], ['Pred: Malignant', 'Pred: Benign'])
plt.yticks([0,1], ['True: Malignant', 'True: Benign'])
plt.title('Confusion Matrix — SVM')
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
print("Classification Report:")
print(classification_report(y_test, y_pred,
      target_names=['Malignant', 'Benign']))

print("\nMetrik individual:")
print(f"Precision : {precision_score(y_test, y_pred):.3f}")
print(f"Recall    : {recall_score(y_test, y_pred):.3f}")
print(f"F1-Score  : {f1_score(y_test, y_pred):.3f}")

### 5.3.1 ROC Curve & AUC

In [ ]:
# ROC Curve — butuh probabilitas atau decision scores
svm_proba = SVC(C=grid_search.best_params_['C'],
                gamma=grid_search.best_params_['gamma'],
                probability=True)
svm_proba.fit(X_train_s, y_train)
y_scores = svm_proba.predict_proba(X_test_s)[:, 1]

fpr, tpr, thresholds = roc_curve(y_test, y_scores)
auc = roc_auc_score(y_test, y_scores)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='blue', linewidth=2, label=f'SVM (AUC = {auc:.3f})')
plt.plot([0,1], [0,1], 'k--', label='Random classifier (AUC = 0.5)')
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve — SVM')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"AUC-ROC: {auc:.4f}")
print("(1.0 = sempurna, 0.5 = random, <0.5 = lebih buruk dari random)")

## 5.4 Metrik untuk Regresi

| Metrik | Formula | Keterangan |
|---|---|---|
| **MSE** | $\frac{1}{n}\sum(y-\hat{y})^2$ | Sensitif terhadap outlier |
| **RMSE** | $\sqrt{MSE}$ | Dalam satuan yang sama dengan y |
| **MAE** | $\frac{1}{n}\sum|y-\hat{y}|$ | Lebih robust terhadap outlier |
| **R²** | $1 - \frac{SS_{res}}{SS_{tot}}$ | Proporsi variansi yang dijelaskan (1=sempurna) |

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.datasets import make_regression

X_reg, y_reg = make_regression(n_samples=200, n_features=10, noise=20, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_reg, y_reg, random_state=42)

model = Ridge().fit(X_tr, y_tr)
y_pred_reg = model.predict(X_te)

mse  = mean_squared_error(y_te, y_pred_reg)
mae  = mean_absolute_error(y_te, y_pred_reg)
r2   = r2_score(y_te, y_pred_reg)

print(f"MSE  : {mse:.2f}")
print(f"RMSE : {np.sqrt(mse):.2f}")
print(f"MAE  : {mae:.2f}")
print(f"R²   : {r2:.4f}")

##  Ringkasan Bab 5

| Konsep | Penjelasan |
|---|---|
| **Cross-Validation** | Estimasi performa yang lebih reliable dari single split |
| **Stratified K-Fold** | CV yang mempertahankan proporsi kelas di tiap fold |
| **GridSearchCV** | Cari hyperparameter terbaik dengan exhaustive search |
| **RandomizedSearchCV** | Hyperparameter search yang lebih efisien |
| **Confusion Matrix** | Visualisasi TP, FP, TN, FN |
| **Precision** | Akurasi prediksi positif (hindari false alarm) |
| **Recall** | Kemampuan mendeteksi kasus positif (hindari miss) |
| **F1-Score** | Keseimbangan Precision dan Recall |
| **AUC-ROC** | Performa model di berbagai threshold (1=sempurna) |

---
 **Next:** Chapter 6 — Algorithm Chains and Pipelines